# 07 · Dados Climáticos e Ambientais em Grade

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/general/07_gridded_climate_environment.pt.ipynb)

**Objetivo de aprendizagem**: aprender a obter conjuntos de dados climáticos e ambientais externos em grade — precipitação (CHIRPS), reanálise climática (ERA5-Land, regional e global), estresse de seca (PDSI) e poluição do ar (GHAP, MERRA-2) —, recortá-los exatamente sobre a grade de pixels de um mapa LCZ, e relacioná-los às classes de Zona Climática Local para entender como a forma urbana molda a exposição climática local.

> **Padrões de plotagem:** toda função que gera figuras retorna por padrão uma `Figure` estática do matplotlib — exiba-a com `display(fig)`. Use `renderer="plotly"` para a figura interativa anterior do Plotly.

## Por que trazer dados climáticos em grade para a grade LCZ?

O esquema de Zonas Climáticas Locais (LCZ) (Stewart & Oke, 2012) classifica o uso e cobertura do solo urbano e natural em 17 classes com base na estrutura da superfície, cobertura e propriedades dos materiais — alta densidade compacta, baixa densidade aberta, árvores densas, água, etc. Isoladamente, um mapa LCZ informa sobre a *forma* urbana. Ele se torna um diagnóstico de clima urbano ao ser sobreposto a dados ambientais independentes em grade: precipitação, temperatura, umidade, estresse de seca e poluição do ar. Cada uma dessas grades interage com a forma urbana de maneiras bem documentadas:

- **Precipitação** interage com a impermeabilização: LCZ 1-3 (alta densidade compacta) têm muito menos superfície permeável do que LCZ 6/9 (baixa densidade aberta) ou LCZ A-D (terreno natural), então um evento de chuva do CHIRPS produz escoamento superficial e risco de enchente relâmpago muito diferentes dependendo da LCZ, mesmo que a chuva em si seja quase uniforme espacialmente na resolução de ~5 km.
- **Temperatura/umidade de reanálise** (ERA5-Land) está em uma grade mais grosseira (~10 km, ou ~9 km na resolução nativa do produto global) do que a própria classificação LCZ, mas ainda é a referência padrão contra a qual sinais locais de ilha de calor urbana (UHI) são medidos — a anomalia entre observações de estação ou satélite e o "background" climático do ERA5 é um diagnóstico comum de UHI (veja a série de notebooks local/ para UHI baseada em estações).
- **Índices de seca** (PDSI, do TerraClimate ou NOAA PSL) integram precipitação e evapotranspiração potencial ao longo do tempo. Como classes LCZ impermeáveis reduzem a infiltração e aumentam o escoamento, o mesmo sinal de seca pode se traduzir em estresse hídrico mais agudo para vegetação urbana e infraestrutura verde em classes LCZ com baixa fração permeável.
- **Poluição do ar** (PM2.5/O3/CO do GHAP, aerossóis do MERRA-2) é moldada pela forma urbana por dois mecanismos opostos: classes LCZ densamente construídas (1-3, 8-10) tendem a reter poluentes por ventilação reduzida (alto fator de visão do céu obstruído, efeitos de "cânion de rua"), enquanto classes LCZ com mais vegetação (A-D) tanto diluem as concentrações de poluentes quanto removem fisicamente partículas do ar.

Anjos et al. (2025, *Scientific Reports*, https://www.nature.com/articles/s41598-025-92000-0) — o artigo por trás do conjunto de ferramentas LCZ4r/LCZ4py — argumenta que essas camadas ambientais só são interpretáveis no contexto da estrutura de superfície sobre a qual estão sobrepostas; um raster bruto de temperatura ou PM2.5 diz pouco sobre *por que* um ponto quente existe até que se saiba se ele está sobre a LCZ 2 (média densidade compacta) ou a LCZ D (plantas baixas).

Este notebook cobre a **família de funções `lcz_grid_*`**, que seguem todas o mesmo contrato: dado um GeoTIFF de mapa LCZ (de `lcz_get_map` ou uma variante), elas baixam um conjunto de dados externo em grade, recortam e reprojetam-no exatamente sobre a grade de pixels do mapa LCZ (mesma forma, transformação, CRS), e retornam um `LCZGridResult` — um array multibanda com NaN fora do footprint válido do mapa LCZ. Como toda saída de `lcz_grid_*` já está alinhada por pixel ao mapa LCZ, nenhuma junção espacial separada é necessária para relacioná-las: `lcz_cal_indexes` (introduzida conceitualmente aqui, coberta função a função no notebook `05_spectral_indices`) e `plot_lcz_relationship` deste notebook funcionam diretamente sobre essa grade compartilhada.

In [ ]:
!pip install LCZ4py

## Mapa de referência

Começamos com um mapa LCZ pequeno e rápido de baixar, usando `lcz_get_map`. **Juiz de Fora** (Liechtenstein) mantém a caixa delimitadora minúscula, o que importa para todo download deste notebook. Uma função (`lcz_grid_era5`) precisa de uma área na América Latina devido à cobertura de sua fonte de dados, então trocamos de cidade apenas nessa célula — indicado claramente quando chegarmos lá.

In [2]:
from LCZ4py import lcz_get_map

map_path = lcz_get_map(city="Juiz de Fora")
print(map_path)

06:43:50 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


/Users/co2map/.lcz4r_cache/clipped_a8db9faa23e32c3b.tif


## `lcz_grid_chirps` — precipitação CHIRPS

O CHIRPS (Funk et al. 2015) é um conjunto de dados de chuva quase global de ~5 km diário/mensal/anual (1981-presente, UCSB Climate Hazards Center), sem necessidade de autenticação. Parâmetros principais:

- `resolution`: `"monthly"` (mm/mês, padrão), `"daily"` (mm/dia), ou `"annual"` (mm/ano).
- `years`, `months`: quais períodos baixar; padrão são os últimos dois anos completos / todos os 12 meses.
- `resampling`: como os pixels de origem são reamostrados na grade do mapa LCZ — `"average"` (padrão, melhor para um campo de chuva de grosseiro para fino), `"bilinear"`, ou `"nearest"`.

Retorna um `LCZGridResult` cujo `.array` é `(n_periods, H, W)` float32 em mm, NaN fora dos pixels válidos do mapa LCZ, e `.bands` dá a data ISO de cada camada. Solicitamos um ano de dados mensais, pequeno o suficiente para executar ao vivo.

In [3]:
from LCZ4py import lcz_grid_chirps

chirps = lcz_grid_chirps(map_path, resolution="monthly", years=[2023], isave=True)
print(chirps.array.shape, chirps.bands[:3], "...", chirps.bands[-3:])
print("units:", chirps.units)

06:43:50 - LCZ4py.general.lcz_grid_chirps - INFO - Saved: /Users/co2map/Documents/lcz4r_python/notebooks/general/LCZ4r_output/lcz_grid_chirps.tif


Using cached CHIRPS stack.
Complete: CHIRPS stack with 12 band(s), cropped to the LCZ map.
(12, 120, 131) ['2023-01-01', '2023-02-01', '2023-03-01'] ... ['2023-10-01', '2023-11-01', '2023-12-01']
units: mm


**Interpretando a saída**: `chirps.array.shape` é `(12, H, W)` — uma banda por mês de 2023. Os valores estão em mm/mês; para uma pequena cidade alpina como Juiz de Fora, deve-se observar um ciclo sazonal (tipicamente meses de verão mais chuvosos). Pixels NaN caem fora do footprint do mapa LCZ. Este é o raster que usaremos abaixo para `plot_grid_only` e `plot_lcz_relationship`.

## `lcz_grid_era5` — clima diário ERA5-Land (espelho América Latina)

A `lcz_grid_era5` baixa agregados diários do ERA5-Land de um espelho no Zenodo (Saldanha, 1950-2025, ~10 km) — sem necessidade de autenticação, **mas sua cobertura é genuinamente restrita à América Latina** (o espelho só possui dados para aproximadamente lon -119..-34, lat -57..34). Chamá-la em um mapa fora dessa janela levanta um `ValueError` claro apontando para `lcz_grid_era5_global`, em vez de falhar profundamente dentro de um erro de extração "sem dados".

Como Juiz de Fora fica na Europa Central, trocamos para **Montevidéu, Uruguai** apenas para esta função. Parâmetros principais:

- `years` (obrigatório), `months`: período a importar.
- `vars`: alias(es) de variável ou `"all"` — opções são `t2m`, `t2m_max`, `t2m_min`, `td2m` (ponto de orvalho), `u10`/`v10` (componentes do vento), `sp` (pressão à superfície), `tp` (precipitação). Padrão `("t2m", "tp")`.
- `resampling`: `"average"` (padrão), `"bilinear"`, ou `"nearest"`.

Conversões de unidade são aplicadas automaticamente: temperatura K→°C, precipitação m→mm, pressão Pa→hPa.

In [4]:
from LCZ4py import lcz_get_map, lcz_grid_era5

# Coverage limitation: lcz_grid_era5's Zenodo mirror is Latin-America-only,
# so we use Montevideo here instead of Juiz de Fora.
map_path_latam = lcz_get_map(city="Montevideo")

era5 = lcz_grid_era5(map_path_latam, years=[2023], months=[1, 2], vars=("t2m", "tp"))
print(era5.array.shape, era5.bands[:3])
print("variables:", era5.variables, "| units:", era5.units)

06:43:50 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


Using cached ERA5-Land stack.
Complete: ERA5-Land stack with 118 band(s), cropped to the LCZ map.
(118, 264, 462) ['t2m_2023-01-01', 't2m_2023-01-02', 't2m_2023-01-03']
variables: ['t2m', 'tp'] | units: mixed


**Interpretando a saída**: as bandas são rotuladas como `"{variável}_{data}"` (ex.: `t2m_2023-01-01`), uma banda por variável por dia. `era5.array.shape` é `(n_vars * n_days, H, W)`. Compare os valores de `t2m` com a chuva do CHIRPS acima conceitualmente: o ERA5-Land fornece o contexto atmosférico (temperatura, precipitação) em uma grade mais grosseira de ~10 km, que as classes de maior resolução do mapa LCZ modulam localmente (microclima urbano vs. rural).

## `lcz_grid_era5_global` — ERA5-Land global via Copernicus CDS

> **Credenciais necessárias — chave de API CDS gratuita**
>
> A `lcz_grid_era5_global` cobre **qualquer região do planeta** (diferente do espelho restrito à América Latina de `lcz_grid_era5`), mas obtém dados ao vivo do Copernicus Climate Data Store (CDS), que exige uma conta gratuita e uma chave de API:
>
> 1. Registre-se em <https://cds.climate.copernicus.eu> (gratuito).
> 2. Vá à sua página de perfil → **API keys** para encontrar sua chave pessoal (uma string UID:chave).
> 3. Veja o guia oficial: <https://cds.climate.copernicus.eu/api-how-to>.
> 4. Passe a chave via `cds_key=` (mostrado abaixo), ou defina as variáveis de ambiente `CDSAPI_KEY`/`CDSAPI_URL`, ou escreva um arquivo `~/.cdsapirc` — qualquer uma das três opções funciona.
>
> **NÃO executamos a célula abaixo** neste notebook (nenhuma chave está configurada neste ambiente), mas o código é real e está pronto para rodar assim que você fornecer sua própria chave. A requisição ao CDS é automaticamente restrita à caixa delimitadora do mapa LCZ (mais uma pequena margem), então mesmo uma execução real nunca transfere dados globais.

Parâmetros principais (apenas granularidade mensal — para dados diários sobre a América Latina, use `lcz_grid_era5`):

- `years` (obrigatório), `months`.
- `vars`: `t2m`, `td2m`, `u10`, `v10`, `sp`, `tp`, `skt` (temperatura da pele), ou `"all"`. (Sem `t2m_max`/`t2m_min` — o produto de médias mensais do CDS só oferece a média mensal.)
- `cds_key`, `cds_url`: credenciais (ou deixe `None` para ler de variáveis de ambiente / `~/.cdsapirc`).

In [ ]:
# NOT EXECUTED in this notebook: requires a real CDS API key.
# Replace "<your-CDS-key>" with the UID:key string from your CDS profile page,
# or omit cds_key/cds_url entirely if you have a ~/.cdsapirc file configured.
from LCZ4py import lcz_grid_era5_global

era5_global = lcz_grid_era5_global(
    map_path,
    years=[2023],
    months=[1, 2],
    vars=("t2m", "tp"),
    cds_key="<your-CDS-key>",
    cds_url="https://cds.climate.copernicus.eu/api",
)
print(era5_global.array.shape, era5_global.bands[:3])

*(Ignorado: nenhuma chave CDS disponível neste ambiente. Com uma chave válida, isso retorna a mesma estrutura `LCZGridResult` de `lcz_grid_era5`, mas utilizável em Juiz de Fora — ou qualquer cidade do mundo — em vez de restrito à América Latina.)*

## `lcz_grid_pdsi` — Índice de Severidade de Seca de Palmer

O PDSI integra precipitação e evapotranspiração potencial em um único índice de estresse hídrico: valores ≥ +4 indicam condições extremamente úmidas, -2 a +2 quase normais, e ≤ -4 seca extrema. A `lcz_grid_pdsi` baixa um produto PDSI pré-calculado e o recorta na grade LCZ. Parâmetros principais:

- `source`: `"terraclimate"` (padrão, ~4 km, 1950-2025, Abatzoglou et al. 2018) ou `"noaa_psl"` (2,5°, 1850-2018, Dai 2011 — mais grosseiro, mas um registro muito mais longo).
- `years`, `months`: padrão são os últimos dois anos completos / todos os 12 meses.
- `resampling`: `"average"` (padrão), `"bilinear"`, `"nearest"`.

Nenhuma autenticação é necessária para nenhuma das fontes.

In [6]:
from LCZ4py import lcz_grid_pdsi

pdsi = lcz_grid_pdsi(map_path, years=[2023], source="terraclimate", isave=True)
print(pdsi.array.shape, pdsi.bands[:3])
print("variables:", pdsi.variables, "| units:", pdsi.units)

06:43:50 - LCZ4py.general.lcz_grid_pdsi - INFO - Saved: /Users/co2map/Documents/lcz4r_python/notebooks/general/LCZ4r_output/lcz_grid_pdsi.tif


Using cached PDSI stack.
Complete: PDSI stack with 12 band(s), cropped to the LCZ map.
(12, 120, 131) ['2023-01-01', '2023-02-01', '2023-03-01']
variables: ['pdsi'] | units: index


**Interpretando a saída**: cada banda é um instantâneo mensal do PDSI. Valores próximos de 0 significam condições de umidade quase normais; valores negativos persistentes ao longo dos meses indicariam estresse de seca que, combinado com a série de chuva do CHIRPS acima, ajuda a distinguir se um período seco é causado por um déficit de chuva de curto prazo ou por um déficit acumulado de balanço hídrico mais longo.

## `lcz_grid_pollution_ghap` — poluição do ar GHAP

O GHAP (Global High Air Pollutants, Wei et al.) fornece campos de poluentes ao nível do solo gerados por IA: PM2.5 (1 km, diário/mensal/anual, 2017-2022), O3 (10 km, anual, 2000-2020), CO (1 km, anual, 2019-2022), hospedado no Zenodo — sem necessidade de autenticação. Parâmetros principais:

- `pollutants`: `"pm25"` (padrão), `"o3"`, `"co"`, ou `"all"`. O3/CO caem silenciosamente para resolução `"annual"` independentemente do que for solicitado.
- `resolution`: `"daily"`/`"monthly"` (apenas PM2.5) ou `"annual"` (todos os poluentes).
- `years`, `months`: padrão são todos os anos disponíveis por poluente/resolução.

In [7]:
from LCZ4py import lcz_grid_pollution_ghap

ghap = lcz_grid_pollution_ghap(map_path, pollutants="pm25", resolution="monthly", years=[2020], months=[6, 7, 8])
print(ghap.array.shape, ghap.bands)
print("units:", ghap.units)

Using cached GHAP stack.
Complete: GHAP stack with 3 band(s), cropped to the LCZ map.
(3, 120, 131) ['pm25_2020-06-01', 'pm25_2020-07-01', 'pm25_2020-08-01']
units: ug_m3


**Interpretando a saída**: as bandas são `"pm25_{data}"`, uma por mês solicitado. Valores mais altos de PM2.5 (µg/m³) concentrados sobre classes LCZ compactas/densas (1-3, 8-10) em comparação com valores mais baixos sobre classes vegetadas (A-D) seriam consistentes com os mecanismos de retenção em cânions urbanos vs. remoção por vegetação descritos na introdução.

## `lcz_grid_pollution_merra2` — aerossóis MERRA-2

> **Credenciais necessárias — login gratuito NASA Earthdata**
>
> A `lcz_grid_pollution_merra2` baixa a reanálise de aerossóis MERRA-2 da NASA (Gelaro et al. 2017) do arquivo GES DISC, que exige uma conta gratuita NASA Earthdata:
>
> 1. Registre-se em <https://urs.earthdata.nasa.gov/users/new> (gratuito).
> 2. Use seu **usuário/senha** do Earthdata diretamente, ou gere um arquivo `.netrc` (veja a documentação do NASA Earthdata Login para o formato exato) e aponte `netrc_path` para ele.
> 3. Passe as credenciais via `earthdata_user=`/`earthdata_pass=` (mostrado abaixo), ou defina as variáveis de ambiente `EARTHDATA_USER`/`EARTHDATA_PASSWORD`.
>
> **NÃO executamos a célula abaixo** (nenhum login Earthdata está configurado neste ambiente), mas o código é real e está pronto para rodar com suas próprias credenciais.

Parâmetros principais:

- `pollutants`: subconjunto de `("pm25", "aod", "so2")`. PM2.5 é derivado de cinco espécies de concentração de massa de aerossóis; AOD é a profundidade óptica total de aerossóis; SO2 é apenas mensal.
- `resolution`: `"monthly"` (padrão) ou `"daily"` — note que o caminho diário do MERRA-2 não está implementado no arquivo público do GES DISC, então ambas as resoluções atualmente retornam o mesmo arquivo de média mensal (limitação documentada, portada tal como está do pacote R para paridade).
- `earthdata_user`, `earthdata_pass`, `netrc_path`: autenticação.

In [ ]:
# NOT EXECUTED in this notebook: requires real NASA Earthdata credentials.
from LCZ4py import lcz_grid_pollution_merra2

merra2 = lcz_grid_pollution_merra2(
    map_path,
    pollutants=("pm25", "aod"),
    resolution="monthly",
    years=[2022],
    months=[6, 7, 8],
    earthdata_user="<your-earthdata-username>",
    earthdata_pass="<your-earthdata-password>",
)
print(merra2.array.shape, merra2.bands)

*(Ignorado: nenhum login Earthdata configurado neste ambiente. Com credenciais válidas, isso retorna a mesma estrutura `LCZGridResult` de `lcz_grid_pollution_ghap`, fornecendo uma segunda fonte independente de aerossóis/poluição para verificar cruzadamente com o GHAP para a mesma cidade/período.)*

## `plot_grid_only` — visualização estática de grade/malha

A `plot_grid_only` renderiza qualquer `GeoDataFrame` de grade/malha vetorial (polígonos ou pontos) sobre um mapa base OpenStreetMap usando Matplotlib + contextily — útil para uma visualização estática rápida de limites municipais, polígonos de zonas LCZ, ou (como aqui) a grade de pixels de uma saída raster `lcz_grid_*`. Parâmetros principais:

- `color_by`: nome da coluna para colorir as feições (`None` para um preenchimento único sem cor).
- `cmap`: colormap do Matplotlib para `color_by`. Padrão `"viridis"`.
- `add_basemap`: sobrepor um mapa base OSM via contextily (ignorado graciosamente se contextily não estiver instalado).

Como `plot_grid_only` espera um `GeoDataFrame` em vez de um array raster bruto, vetorizamos uma banda da saída CHIRPS salva acima (`isave=True` a gravou em `LCZ4r_output/lcz_grid_chirps.tif`) em polígonos com `rasterio.features.shapes`, e então plotamos.

In [ ]:
from IPython.display import display

import geopandas as gpd
import numpy as np
import rasterio
from rasterio.features import shapes
from shapely.geometry import shape

from LCZ4py import plot_grid_only

with rasterio.open("LCZ4r_output/lcz_grid_chirps.tif") as src:
    band = src.read(1)
    transform = src.transform
    crs = src.crs

valid_mask = ~np.isnan(band)
polygons, values = [], []
for geom, val in shapes(band, mask=valid_mask, transform=transform):
    polygons.append(shape(geom))
    values.append(val)

grid_gdf = gpd.GeoDataFrame({"precip_mm": values}, geometry=polygons, crs=crs)
fig = plot_grid_only(grid_gdf, color_by="precip_mm", cmap="Blues", title="CHIRPS monthly precipitation (Jan 2023)")
display(fig)

**Interpretando a saída**: cada polígono é um pixel do CHIRPS reprojetado na grade LCZ, colorido pela chuva de janeiro de 2023 (mm). Como o raster subjacente foi mascarado para o footprint válido do mapa LCZ, apenas os pixels dentro do limite da cidade são mostrados — você está observando o padrão espacial real da chuva ao qual as classes LCZ abaixo estão expostas.

## `plot_lcz_relationship` — distribuição da variável por classe LCZ

A `plot_lcz_relationship` é a contraparte visual rápida de `lcz_cal_indexes` (coberta no notebook `05_spectral_indices`, e a ferramenta geral para resumir *qualquer* variável em grade por classe LCZ em forma de tabela). Ela lê um GeoTIFF de classificação LCZ e um raster ambiental de banda única (reprojetando este último na grade do primeiro, se necessário), e então plota a distribuição da variável agrupada por classe LCZ. Parâmetros principais:

- `variable_name`: rótulo do eixo para o valor.
- `plot_type`: `"box"`/`"violin"` (distribuição por classe), `"scatter"` (por pixel, com jitter), ou `"heatmap"` (um valor agregado por classe).
- `agg_fun`: agregação para `"heatmap"` — `"mean"` (padrão), `"median"`, `"sum"`.

Usamos aqui na pilha de precipitação do CHIRPS (usando uma única banda como raster representativo de banda única) para ver como a exposição à chuva difere por classe LCZ dentro de Juiz de Fora.

## Figuras de grade prontas para publicação com `style`

As funções de plotagem agora aceitam `style="default"`, `style="nature"`, `style="science"` e `style="generic_bw"`. Os presets de revista ajustam a família de fontes, o tamanho físico em milímetros, o DPI (300 para os estilos voltados à publicação) e as cores para que figuras baseadas em grade possam ser exportadas diretamente para um manuscrito.

Para gráficos de grade com aparência de mapa, também há elementos cartográficos automáticos: `add_scalebar=True` e `add_north_arrow=True`.

Um exemplo curto pronto para publicação da camada em grade:

- `style="science"` para a figura de grade.
- `isave=True` com `save_extension="png"` ou `"pdf"` para gravar a imagem final.


In [ ]:
from IPython.display import display

from LCZ4py import plot_grid_only

fig_grid_pub = plot_grid_only(
    grid_gdf,
    color_by="precip_mm",
    title="Grade CHIRPS — estilo de publicação",
    style="science",
    isave=True,
    save_extension="png",
    add_scalebar=True,
    add_north_arrow=True,
)
display(fig_grid_pub)


In [ ]:
import rasterio

from IPython.display import display
from LCZ4py import plot_lcz_relationship

# Extract a single band (January 2023) to a standalone single-band GeoTIFF,
# since plot_lcz_relationship expects a single-band variable raster.
with rasterio.open("LCZ4r_output/lcz_grid_chirps.tif") as src:
    band1 = src.read(1)
    profile = src.profile.copy()
profile.update(count=1)
variable_path = "LCZ4r_output/lcz_grid_chirps_jan2023.tif"
with rasterio.open(variable_path, "w", **profile) as dst:
    dst.write(band1, 1)

fig = plot_lcz_relationship(map_path, variable_path, variable_name="precip_mm (Jan 2023)", plot_type="box")
display(fig)

**Interpretando a saída**: o box plot mostra a distribuição da chuva CHIRPS de janeiro de 2023 para cada classe LCZ presente em Juiz de Fora. Como a precipitação em si não varia muito com a forma urbana na resolução de ~5 km do CHIRPS para uma cidade tão pequena, espera-se que as caixas pareçam amplamente semelhantes entre as classes aqui — a versão mais marcante desse gráfico aparece com uma variável que realmente responde à estrutura urbana, como a pilha de PM2.5 do GHAP acima (tente substituir por `LCZ4r_output/lcz_grid_ghap.tif`, extraindo uma banda da mesma forma, para ver padrões de poluição dependentes da classe). Esse é exatamente o tipo de exploração que `lcz_cal_indexes` (veja `05_spectral_indices`) transforma em uma tabela reutilizável de estatísticas por classe, quando você precisa de números em vez de um gráfico.

## Conclusão e próximos passos

Este notebook cobriu toda a família `lcz_grid_*` para trazer contexto ambiental externo em grade para a grade de pixels de um mapa LCZ:

- **`lcz_grid_chirps`** — precipitação, sem autenticação, global.
- **`lcz_grid_era5`** — clima de reanálise diário, sem autenticação, cobertura restrita à América Latina.
- **`lcz_grid_era5_global`** — clima de reanálise mensal, qualquer região, requer chave de API CDS gratuita.
- **`lcz_grid_pdsi`** — índice de severidade de seca, sem autenticação, global.
- **`lcz_grid_pollution_ghap`** — PM2.5/O3/CO, sem autenticação, global.
- **`lcz_grid_pollution_merra2`** — reanálise de aerossóis, requer login gratuito NASA Earthdata.

Além de dois auxiliares de visualização: **`plot_grid_only`** para uma visualização estática rápida de qualquer grade/malha vetorial, e **`plot_lcz_relationship`** para uma leitura visual rápida de como uma variável em grade se distribui pelas classes LCZ — a contraparte visual da tabular `lcz_cal_indexes` (veja `05_spectral_indices`).

Este é o **último notebook da série `general/`**. Junto com `01_map_acquisition` até `06_urban_canopy_parameters`, você agora tem o kit de ferramentas completo para adquirir, descrever e contextualizar um mapa LCZ para qualquer cidade do planeta. A série continua em **`notebooks/local/01_lcz_time_series`**, que muda da análise raster de mapa inteiro para séries temporais de temperatura do ar baseadas em estações e pontos — o ponto de partida da série local/, cobrindo anomalias de temperatura, intensidade de ilha de calor urbana, interpolação espacial e conforto térmico.